# City-Day AQI Prediction

This notebook estimates daily AQI from city, date, and same-day pollutant readings in `city_day.csv`. It compares six regression models and evaluates the selected model on later dates.

## Problem and Goal

The target is the numeric `AQI` column. `AQI_Bucket` is excluded because it is derived from or closely tied to AQI. The model estimates AQI after same-day pollutant readings are available; it is not a forecast made before those readings exist. We use mean absolute error (MAE) as the main metric because it measures typical error in AQI points. The data file does not include measurement units or a data dictionary, so pollutant descriptions and AQI calculation details cannot be confirmed here.

## Imports

If needed, install the packages with `pip install pandas numpy matplotlib seaborn scikit-learn`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

SEED = 42
sns.set_theme(style="whitegrid")

## 3. Dataset Loading

Load the supplied file and parse the date column. Invalid date values are reported; they are excluded from modeling because calendar features and chronological splitting require valid dates.

In [ ]:
data_path = Path("city_day.csv")
if not data_path.exists():
    raise FileNotFoundError("Put city_day.csv beside this notebook.")

df = pd.read_csv(data_path)
if "Date" not in df.columns or "AQI" not in df.columns:
    raise ValueError("The dataset must contain Date and AQI columns.")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Data types:")
df.info()
display(df.head())
display(df.describe(include="all").T)

## Dataset Summary

Review the column types, summary statistics, and feature groups. The source file does not provide detailed definitions or units.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include="object").columns.tolist()

print("Numeric columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

## Data Quality

Check missing values, duplicate records, target coverage, and category counts.

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Rows missing AQI:", df["AQI"].isna().sum())
print("Rows with invalid dates:", df["Date"].isna().sum())

missing_counts = df.isna().sum().sort_values(ascending=False)
display(missing_counts.to_frame("Missing values"))

print("City counts:")
display(df["City"].value_counts().head(10))
print("AQI bucket counts:")
display(df["AQI_Bucket"].value_counts(dropna=False))

## Exploratory Data Analysis

The plots below show missingness, distributions, correlations, categories, and AQI over time.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
missing_rates = df.isna().mean().sort_values(ascending=False)

ax = missing_rates.plot.bar(figsize=(10, 4), color="steelblue")
ax.set_title("Missing values by feature")
ax.set_ylabel("Fraction missing")
ax.set_xlabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print("Higher bars indicate features that need more imputation.")

In [ ]:
ax = sns.histplot(data=df, x="AQI", bins=40, kde=True)
ax.set_title("Distribution of AQI")
ax.set_xlabel("AQI")
ax.set_ylabel("City-day count")
plt.tight_layout()
plt.show()

print("AQI summary:")
display(df["AQI"].describe().round(2))

In [ ]:
plot_numeric = [column for column in numeric_cols if column != "AQI"]

if plot_numeric:
    df[plot_numeric].hist(
        figsize=(15, 12), bins=30, grid=False, color="slateblue"
    )
    plt.suptitle("Distributions of numeric measurements")
    plt.tight_layout()
    plt.show()

print("Use these plots to inspect skew and unusual ranges.")

In [ ]:
if plot_numeric:
    fig, ax = plt.subplots(figsize=(14, 5))
    sns.boxplot(data=df[plot_numeric], orient="h", showfliers=False, ax=ax)
    ax.set_title("Numeric measurement ranges")
    ax.set_xlabel("Recorded value")
    plt.tight_layout()
    plt.show()

print("Extreme values are kept unless the source confirms they are errors.")

In [ ]:
corr_cols = [
    column for column in numeric_cols
    if df[column].notna().sum() > 1
]
correlations = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(correlations, cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation among numeric fields")
plt.tight_layout()
plt.show()

print("Correlations show associations, not causes.")

In [ ]:
if "City" in df:
    city_counts = df["City"].value_counts().head(20).sort_values()
    ax = city_counts.plot.barh(figsize=(11, 5), color="teal")
    ax.set_title("Records by city (top 20)")
    ax.set_xlabel("Rows")
    ax.set_ylabel("City")
    plt.tight_layout()
    plt.show()
    print("Cities represented:", df["City"].nunique())

if "AQI_Bucket" in df:
    bucket_counts = df["AQI_Bucket"].value_counts(dropna=False)
    ax = bucket_counts.plot.bar(figsize=(9, 4), color="darkorange")
    ax.set_title("AQI bucket counts (missing included)")
    ax.set_xlabel("AQI bucket")
    ax.set_ylabel("Rows")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
    print("AQI_Bucket is excluded from the model inputs to avoid leakage.")

In [ ]:
valid_target = df.dropna(subset=["AQI", "Date"]).copy()
valid_target["Year"] = valid_target["Date"].dt.year
year_aqi = valid_target.groupby("Year")["AQI"].agg(
    count="count", mean="mean", median="median"
)
display(year_aqi)

fig, ax = plt.subplots(figsize=(11, 4))
sns.lineplot(data=valid_target, x="Date", y="AQI", errorbar=None, ax=ax)
ax.set_title("AQI over time (cities pooled)")
ax.set_xlabel("Date")
ax.set_ylabel("AQI")
plt.tight_layout()
plt.show()

print("The later-date holdout helps evaluate performance over time.")

## Prepare Features

Drop duplicate records and rows without a usable target or date. Create simple calendar features. Fit imputers, scaling, and encoding on training data through a scikit-learn pipeline.

In [ ]:
data = df.drop_duplicates().dropna(subset=["AQI", "Date"]).copy()
data = data.sort_values(["Date", "City"]).reset_index(drop=True)

data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month
data["Day"] = data["Date"].dt.day
data["DayOfWeek"] = data["Date"].dt.dayofweek

target = data["AQI"]
feature_columns = [
    column for column in data.columns
    if column not in ["AQI", "AQI_Bucket", "Date"]
]
X = data[feature_columns]

categorical_features = X.select_dtypes(include="object").columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
print("Rows for modeling:", len(X))

## Chronological Train-Test Split

Use the earliest 80% of unique dates for training and the latest 20% for testing. Keeping dates together avoids putting cities from the same day on both sides of the split.

In [ ]:
unique_dates = np.sort(data["Date"].unique())
split_index = int(len(unique_dates) * 0.8)
split_date = pd.Timestamp(unique_dates[split_index])
train_rows = data["Date"] < split_date

X_train = X.loc[train_rows]
X_test = X.loc[~train_rows]
y_train = target.loc[train_rows]
y_test = target.loc[~train_rows]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Test period starts:", split_date.date())

## Baseline

Compare the models with a simple baseline that predicts the training-set median AQI.

In [ ]:
baseline = DummyRegressor(strategy="median")


## Regressors

Compare six common regression algorithms. Random Forest, Extra Trees, and Gradient Boosting are ensemble models. Each model uses the same preprocessing and training data.

In [ ]:
numeric_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_steps, numeric_features),
    ("categorical", categorical_steps, categorical_features),
])

def make_pipeline(model):
    return Pipeline([
        ("preprocessing", clone(preprocessor)),
        ("model", clone(model)),
    ])

models = {
    "Median baseline": baseline,
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=18, min_samples_leaf=5, random_state=SEED),
    "Random Forest": RandomForestRegressor(n_estimators=120, min_samples_leaf=2, random_state=SEED),
    "Extra Trees": ExtraTreesRegressor(n_estimators=120, min_samples_leaf=2, random_state=SEED),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, random_state=SEED),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=12, weights="distance"),
}

## Cross-Validation and Model Comparison

Compare models using MAE and R2 on chronological validation folds from the training period. Each fold keeps all city records for a date together. Use cross-validation to choose a model; reserve the final test period for one evaluation.

In [ ]:
training_dates = data.loc[X_train.index, "Date"].reset_index(drop=True)
date_blocks = np.array_split(np.sort(training_dates.unique()), 5)
cv_splits = []

for block_number in range(1, 5):
    earlier_dates = np.concatenate(date_blocks[:block_number])
    validation_dates = date_blocks[block_number]
    train_indices = np.flatnonzero(training_dates.isin(earlier_dates))
    validation_indices = np.flatnonzero(training_dates.isin(validation_dates))
    cv_splits.append((train_indices, validation_indices))

cv_rows = []
for name, model in models.items():
    fold_mae = []
    fold_r2 = []

    for train_indices, validation_indices in cv_splits:
        pipeline = make_pipeline(model)
        pipeline.fit(X_train.iloc[train_indices], y_train.iloc[train_indices])
        predictions = pipeline.predict(X_train.iloc[validation_indices])
        actual = y_train.iloc[validation_indices]

        fold_mae.append(mean_absolute_error(actual, predictions))
        fold_r2.append(r2_score(actual, predictions))

    cv_rows.append({
        "Model": name,
        "Average validation MAE": np.mean(fold_mae),
        "Average validation R2": np.mean(fold_r2),
    })

cv_results = pd.DataFrame(cv_rows)
cv_results = cv_results.sort_values("Average validation MAE").reset_index(drop=True)
display(cv_results)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(
    cv_results["Model"],
    cv_results["Average validation MAE"],
    color="steelblue",
)
ax.invert_yaxis()
ax.set_title("Chronological cross-validation MAE (lower is better)")
ax.set_xlabel("Mean absolute error in AQI points")
plt.tight_layout()
plt.show()

## Final Model and Test Results

Select the model with the lowest mean validation MAE, refit it on all training data, and evaluate it once on the later-date test set.

In [ ]:
best_name = cv_results.iloc[0]["Model"]
best_model = make_pipeline(models[best_name])
best_model.fit(X_train, y_train)
test_prediction = best_model.predict(X_test)

final_results = pd.DataFrame([{
    "Selected model": best_name,
    "Test MAE": mean_absolute_error(y_test, test_prediction),
    "Test RMSE": mean_squared_error(y_test, test_prediction) ** 0.5,
    "Test R2": r2_score(y_test, test_prediction),
}])
display(final_results)
print("Selected using the lowest average validation MAE:", best_name)

## Final Model Diagnostics

Compare actual and predicted AQI, then inspect residuals for patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, test_prediction, alpha=0.4)
plot_limits = [min(y_test.min(), test_prediction.min()), max(y_test.max(), test_prediction.max())]
ax.plot(plot_limits, plot_limits, "--", color="red")
ax.set_title("Actual and predicted AQI")
ax.set_xlabel("Actual AQI")
ax.set_ylabel("Predicted AQI")
plt.tight_layout()
plt.show()

residuals = y_test - test_prediction
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(test_prediction, residuals, alpha=0.4)
ax.axhline(0, linestyle="--", color="red")
ax.set_title("Residuals")
ax.set_xlabel("Predicted AQI")
ax.set_ylabel("Actual minus predicted AQI")
plt.tight_layout()
plt.show()

## Conclusions and Limitations

Use the computed validation and test metrics above to summarize the results. This split tests later dates for cities represented in the training period; it does not test performance on entirely new cities. Same-day pollutant readings are required inputs. The data source does not establish measurement units, sensor quality, or the AQI formula. Results may change if city coverage, missingness, or pollutant patterns change over time.